# Speech Alignment And Codec Tokenization Pipeline

Audio data preparation notebook.

raw audio + transcript -> Wav2Vec2 CTC emissions -> forced alignment -> word/phrase clips -> EnCodec RVQ tokens -> reconstructions + metadata

No STT training, no TTS training, no voice cloning here.


In [1]:
REPO_URL = 'https://github.com/hkhattak09/AudioAI.git'
REPO_DIR = '/content/AudioAI'
BRANCH = 'main'

import os
import sys
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

cache_root = Path('/content/drive/MyDrive/audioai/cache')
hf_root = cache_root / 'huggingface'
torch_root = cache_root / 'torch'

os.environ['HF_HOME'] = str(hf_root)
os.environ['HF_HUB_CACHE'] = str(hf_root / 'hub')
os.environ['HF_DATASETS_CACHE'] = str(hf_root / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(hf_root / 'transformers')
os.environ['TORCH_HOME'] = str(torch_root)
os.environ['XDG_CACHE_HOME'] = str(cache_root / 'xdg')
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'

for folder in [cache_root,hf_root,hf_root / 'hub',hf_root / 'datasets',hf_root / 'transformers',torch_root,cache_root / 'xdg']:
  folder.mkdir(parents=True,exist_ok=True)

def local_disk(label):
  total,used,free = shutil.disk_usage('/content')
  print(f'{label}: local disk used {used/1024**3:.2f} GB / {total/1024**3:.2f} GB, free {free/1024**3:.2f} GB')

local_disk('before setup')
for stale_cache in ['/root/.cache/pip','/root/.cache/huggingface']:
  stale_cache = Path(stale_cache)
  if stale_cache.exists():
    print('removing local cache =',stale_cache)
    shutil.rmtree(stale_cache,ignore_errors=True)

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
  os.chdir(repo_dir)
  !git fetch --all --prune
  !git checkout {BRANCH}
  !git pull --ff-only
else:
  !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
  os.chdir(repo_dir)

!pip install -q --no-cache-dir -e .
!pip install -q --no-cache-dir "datasets[audio]" transformers accelerate torchaudio soundfile librosa matplotlib encodec

sys.path.insert(0, str(repo_dir / 'src'))
local_disk('after setup')
print('working directory =', Path.cwd())
print('hf cache =', os.environ['HF_HOME'])
print('torch cache =', os.environ['TORCH_HOME'])
print('xdg cache =', os.environ['XDG_CACHE_HOME'])


Mounted at /content/drive
before setup: local disk used 42.85 GB / 112.64 GB, free 69.77 GB
Cloning into '/content/AudioAI'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 85 (delta 7), reused 85 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 37.80 KiB | 3.15 MiB/s, done.
Resolving deltas: 100% (7/7), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for voice-ai (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 44.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
after setup: local disk used 42.86 GB / 112.64 GB, free 69.77 GB
working directory = /content/AudioAI
hf cache = /content/drive/MyDrive/audioai/cache/huggingface
torch cache = /cont

In [2]:
import os
import torch
import torchaudio
import transformers
import datasets
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import json
import re
import time
import shutil
from datetime import datetime
from pathlib import Path
from datasets import Audio, load_dataset
from transformers import AutoProcessor, AutoModelForCTC
from encodec import EncodecModel
from encodec.utils import convert_audio
from voice_ai.stt.aligner import AlignmentError, forced_align_ctc

torch.hub.set_dir(str(Path(os.environ['TORCH_HOME']) / 'hub'))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', DEVICE)
print('torch =', torch.__version__)
print('torchaudio =', torchaudio.__version__)
print('transformers =', transformers.__version__)
print('datasets =', datasets.__version__)
print('encodec = OK')
print('hf cache =', os.environ.get('HF_HOME'))
print('torch hub cache =', torch.hub.get_dir())
if torch.cuda.is_available():
  print('gpu =', torch.cuda.get_device_name(0))
  print('gpu memory GB =', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


device = cuda
torch = 2.10.0+cu128
torchaudio = 2.10.0+cu128
transformers = 5.0.0
datasets = 4.0.0
encodec = OK
hf cache = /content/drive/MyDrive/audioai/cache/huggingface
torch hub cache = /content/drive/MyDrive/audioai/cache/torch/hub
gpu = Tesla T4
gpu memory GB = 14.56


In [3]:
print('drive is mounted at /content/drive')
print('artifacts will be written to /content/drive/MyDrive/audioai')


drive is mounted at /content/drive
artifacts will be written to /content/drive/MyDrive/audioai


### Config

In [4]:
run_profile = 'portfolio_t4'
resume_run_name = None
# resume_run_name = 'portfolio_t4_20260511_143000'

dataset_name = 'openslr/librispeech_asr'
dataset_config = 'clean'
dataset_split = 'validation'
dataset_streaming = True

max_utterances = 300
max_segments = 1000
max_runtime_hours = 5.0
min_free_disk_gb = 8.0
cleanup_local_caches = True

wav2vec2_model_name = 'facebook/wav2vec2-base-960h'
asr_sample_rate = 16000

encodec_bandwidth = 6.0
encodec_sample_rate = 24000

min_segment_sec = 0.4
max_segment_sec = 8.0
target_segment_sec = 3.0
padding_sec = 0.05
min_word_confidence = 0.01

# smoke config
# run_profile = 'smoke_t4'
# dataset_split = 'validation'
# max_utterances = 10
# max_segments = 50
# max_runtime_hours = 1.0

# larger config if the 300/1000 run finishes comfortably
# run_profile = 'portfolio_t4_larger'
# dataset_split = 'validation'
# max_utterances = 500
# max_segments = 1500
# max_runtime_hours = 5.0

runs_root = Path('/content/drive/MyDrive/audioai/alignment_codec_runs')
if resume_run_name:
  run_name = resume_run_name
else:
  run_name = f"{run_profile}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_root = runs_root / run_name
print('run_name =', run_name)
print('run_root =', run_root)
print('dataset_streaming =', dataset_streaming)


run_name = portfolio_t4_20260511_185544
run_root = /content/drive/MyDrive/audioai/alignment_codec_runs/portfolio_t4_20260511_185544
dataset_streaming = True


In [5]:
align_dir = run_root / 'alignments'
seg_wav_dir = run_root / 'segments' / 'wav'
seg_meta_dir = run_root / 'segments' / 'meta'
codec_token_dir = run_root / 'codec' / 'tokens'
codec_recon_dir = run_root / 'codec' / 'reconstructions'
codec_meta_dir = run_root / 'codec' / 'meta'
report_dir = run_root / 'reports'
sample_dir = report_dir / 'samples'

for folder in [run_root,align_dir,seg_wav_dir,seg_meta_dir,codec_token_dir,codec_recon_dir,codec_meta_dir,report_dir,sample_dir]:
  folder.mkdir(parents=True,exist_ok=True)

start_file = run_root / 'RUN_STARTED.txt'
resume_file = run_root / f"RUN_RESUMED_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
run_note = (
  f'run = {run_name}\n'
  f'profile = {run_profile}\n'
  f'wav2vec2 = {wav2vec2_model_name}\n'
  f'encodec = encodec_24khz at {encodec_bandwidth} kbps\n'
  f'no model training is performed in this notebook\n'
)
if resume_run_name and start_file.exists():
  resume_file.write_text('resumed at ' + datetime.now().isoformat() + '\n' + run_note)
else:
  start_file.write_text('started at ' + datetime.now().isoformat() + '\n' + run_note)

(run_root / 'README.md').write_text(
  '# Speech Alignment And Codec Tokenization Run\n\n'
  f'Run: {run_name}\n\n'
  f'Wav2Vec2 CTC: {wav2vec2_model_name} pretrained, not trained here.\n\n'
  f'EnCodec: encodec_24khz at {encodec_bandwidth} kbps pretrained, not trained here.\n\n'
  'This run creates word-aligned speech clips, EnCodec RVQ tokens, waveform reconstructions, and metadata for downstream TTS/ASR training.\n'
)


357

### Utility functions

In [6]:
run_start = time.time()

def elapsed_hours():
  return (time.time() - run_start) / 3600

def disk_free_gb(path='/content'):
  total,used,free = shutil.disk_usage(path)
  return free / 1024**3

def show_disk(label):
  total,used,free = shutil.disk_usage('/content')
  print(f'{label}: local disk used {used/1024**3:.2f} GB / {total/1024**3:.2f} GB, free {free/1024**3:.2f} GB')

def disk_low():
  return disk_free_gb('/content') < min_free_disk_gb

def time_over():
  return elapsed_hours() >= max_runtime_hours

def should_stop():
  return time_over() or disk_low()

def cleanup_cache():
  if not cleanup_local_caches:
    return
  for path in ['/root/.cache/pip','/root/.cache/huggingface']:
    path = Path(path)
    if path.exists():
      print('removing local cache =',path)
      shutil.rmtree(path,ignore_errors=True)

def write_json(path,obj):
  path = Path(path)
  path.parent.mkdir(parents=True,exist_ok=True)
  path.write_text(json.dumps(obj,indent=2,ensure_ascii=False))

def read_json(path):
  return json.loads(Path(path).read_text())

def rebuild_jsonl(meta_dir,out_path):
  records = []
  meta_dir = Path(meta_dir)
  for file in sorted(meta_dir.glob('*.json')):
    records.append(read_json(file))
  out_path = Path(out_path)
  out_path.parent.mkdir(parents=True,exist_ok=True)
  out_path.write_text('\n'.join(json.dumps(x,ensure_ascii=False) for x in records) + ('\n' if records else ''))
  return len(records)

def duration(waveform,sr):
  if waveform.numel() == 0:
    return 0.0
  return waveform.shape[-1] / float(sr)

def audio_stats(waveform):
  if waveform.numel() == 0:
    return {'rms':0.0,'peak':0.0,'finite':True}
  x = waveform.reshape(-1).float()
  return {
    'rms':float(x.pow(2).mean().sqrt().item()),
    'peak':float(x.abs().max().item()),
    'finite':bool(torch.isfinite(x).all().item()),
  }

def save_wav(path,waveform,sr):
  path = Path(path)
  path.parent.mkdir(parents=True,exist_ok=True)
  if waveform.dim() == 1:
    waveform = waveform.unsqueeze(0)
  torchaudio.save(str(path),waveform.cpu(),sr)

def get_text(row):
  for key in ['text','sentence','normalized_text','transcript']:
    if key in row and row[key]:
      return str(row[key])
  raise KeyError(f'no transcript field found. keys = {list(row.keys())}')

def audio_id(index,row):
  for key in ['id','utterance_id','file_id']:
    if key in row and row[key] is not None:
      return str(row[key])
  return f'utt_{index:06d}'

def normalize_text(text):
  text = text.upper()
  text = re.sub(r"[^A-Z' ]+",' ',text)
  text = re.sub(r'\s+',' ',text).strip()
  return text

def ctc_text(text):
  return text.replace(' ','|')

show_disk('start')
cleanup_cache()
show_disk('after local cache cleanup')


start: local disk used 42.90 GB / 112.64 GB, free 69.72 GB
after local cache cleanup: local disk used 42.90 GB / 112.64 GB, free 69.72 GB


### Loading data

In [7]:
dataset_load_split = dataset_split.split('[')[0] if dataset_streaming else dataset_split
raw_ds = load_dataset(dataset_name,dataset_config,split=dataset_load_split,streaming=dataset_streaming)
raw_ds = raw_ds.cast_column('audio',Audio(sampling_rate=asr_sample_rate))

raw_items = []
for i,row in enumerate(raw_ds):
  if len(raw_items) >= max_utterances:
    break
  if should_stop():
    print('stopping while loading data because time or disk limit was reached')
    break

  audio = row['audio']
  waveform = torch.tensor(audio['array'],dtype=torch.float32)
  if waveform.dim() > 1:
    waveform = waveform.mean(dim=0)

  original = get_text(row)
  normal = normalize_text(original)
  if not normal:
    print('skipping empty transcript',i)
    continue

  raw_items.append({
    'raw_item_index':len(raw_items),
    'source_row_index':i,
    'utterance_id':audio_id(i,row),
    'waveform':waveform,
    'original_text':original,
    'normalized_text':normal,
    'duration_sec':duration(waveform,asr_sample_rate),
    'sample_rate':asr_sample_rate,
    'num_samples':int(waveform.shape[-1]),
  })

print('requested split =',dataset_split)
print('loaded split =',dataset_load_split)
print('streaming =',dataset_streaming)
print('raw items =',len(raw_items))
show_disk('after data load')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

requested split = validation
loaded split = validation
streaming = True
raw items = 300
after data load: local disk used 42.90 GB / 112.64 GB, free 69.72 GB


In [8]:
manifest = []
for item in raw_items:
  record = {
    'utterance_id':item['utterance_id'],
    'dataset_name':dataset_name,
    'dataset_config':dataset_config,
    'dataset_split':dataset_split,
    'dataset_load_split':dataset_load_split,
    'dataset_streaming':dataset_streaming,
    'row_index':item['source_row_index'],
    'raw_item_index':item['raw_item_index'],
    'original_text':item['original_text'],
    'normalized_text':item['normalized_text'],
    'duration_sec':item['duration_sec'],
    'sample_rate':item['sample_rate'],
    'num_samples':item['num_samples'],
  }
  manifest.append(record)

raw_manifest = run_root / 'raw_manifest.jsonl'
raw_manifest.write_text('\n'.join(json.dumps(x,ensure_ascii=False) for x in manifest) + ('\n' if manifest else ''))
print(f'usable rows = {len(manifest)}')
print('raw_manifest =',raw_manifest)


usable rows = 300
raw_manifest = /content/drive/MyDrive/audioai/alignment_codec_runs/portfolio_t4_20260511_185544/raw_manifest.jsonl


### Wav2Vec2 emissions and forced alignment

In [9]:
processor = AutoProcessor.from_pretrained(wav2vec2_model_name)
asr_model = AutoModelForCTC.from_pretrained(wav2vec2_model_name).to(DEVICE)
asr_model.eval()

vocab = processor.tokenizer.get_vocab()
id_to_token = {v:k for k,v in vocab.items()}
blank_id = processor.tokenizer.pad_token_id
unk_id = processor.tokenizer.unk_token_id

print('model =',wav2vec2_model_name)
print('vocab size =',len(vocab))
print('blank_id =',blank_id)


Loading weights:   0%|          | 0/212 [00:02<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model = facebook/wav2vec2-base-960h
vocab size = 32
blank_id = 0


In [10]:
aligned = 0
failed = 0

for i,record in enumerate(manifest):
  if should_stop():
    print('time or disk limit hit during alignment')
    break

  uid = record['utterance_id']
  align_path = align_dir / f'{uid}.json'

  if align_path.exists():
    old = read_json(align_path)
    if old.get('status') == 'aligned':
      aligned += 1
      continue
    if old.get('status') == 'failed':
      failed += 1
      continue

  item = raw_items[record['raw_item_index']]
  waveform = item['waveform']

  audio_dur = duration(waveform,asr_sample_rate)
  normal = record['normalized_text']
  tokenizer_text = ctc_text(normal)
  tokens = processor.tokenizer(tokenizer_text,add_special_tokens=False).input_ids

  status = 'failed'
  error = None
  alignment_dict = None
  n_frames = 0
  vocab_size = 0

  try:
    if len(tokens) == 0:
      raise AlignmentError('empty_token_sequence')
    if unk_id is not None and unk_id in tokens:
      raise AlignmentError('unknown_token_in_transcript')

    inputs = processor(waveform.numpy(),sampling_rate=asr_sample_rate,return_tensors='pt')
    with torch.no_grad():
      logits = asr_model(inputs.input_values.to(DEVICE)).logits[0]
    emissions = torch.log_softmax(logits,dim=-1).detach().cpu()

    n_frames = int(emissions.shape[0])
    vocab_size = int(emissions.shape[1])
    if len(tokens) >= n_frames:
      raise AlignmentError('text_too_long_for_emissions')

    alignment = forced_align_ctc(
      emission=emissions,
      tokens=tokens,
      id_to_token=id_to_token,
      blank_id=blank_id,
      duration_sec=audio_dur,
      word_delimiter='|',
    )
    alignment_dict = alignment.to_dict()
    status = 'aligned'
    aligned += 1
  except Exception as e:
    error = str(e)
    failed += 1

  out = {
    'utterance_id':uid,
    'row_index':record['row_index'],
    'original_text':record['original_text'],
    'normalized_text':normal,
    'ctc_text':tokenizer_text,
    'duration_sec':audio_dur,
    'emission_metadata':{
      'num_emission_frames':n_frames,
      'emission_vocab_size':vocab_size,
      'seconds_per_frame':audio_dur / n_frames if n_frames else 0.0,
      'blank_id':blank_id,
      'model_name':wav2vec2_model_name,
    },
    'token_spans':alignment_dict['token_spans'] if alignment_dict else [],
    'word_spans':alignment_dict['word_spans'] if alignment_dict else [],
    'average_word_confidence':alignment_dict['average_word_confidence'] if alignment_dict else 0.0,
    'status':status,
    'error_reason':error,
  }
  write_json(align_path,out)

  if (i+1)%25 == 0 or status == 'failed':
    print(f'{i+1}/{len(manifest)} {uid} {status}' + (f' - {error}' if error else ''))

print(f'aligned = {aligned}, failed = {failed}')


25/300 2277-149896-0024 aligned
50/300 2277-149897-0014 aligned
75/300 2277-149874-0001 aligned
100/300 2035-147960-0004 aligned
125/300 2035-152373-0012 aligned
150/300 2035-147961-0018 aligned
175/300 2086-149214-0002 aligned
200/300 2086-149220-0022 aligned
225/300 2086-149220-0047 aligned
250/300 7976-110124-0022 aligned
275/300 7976-105575-0021 aligned
300/300 7976-110523-0016 aligned
aligned = 300, failed = 0


### Creating segments

In [11]:
total_segments = 0

for i,record in enumerate(manifest):
  if should_stop():
    print('time or disk limit hit during segmentation')
    break
  if total_segments >= max_segments:
    print('max segments reached')
    break

  uid = record['utterance_id']
  align_path = align_dir / f'{uid}.json'
  if not align_path.exists():
    continue

  align = read_json(align_path)
  if align.get('status') != 'aligned':
    continue

  words = align.get('word_spans',[])
  if not words:
    continue

  item = raw_items[record['raw_item_index']]
  waveform = item['waveform']
  utt_dur = duration(waveform,asr_sample_rate)

  groups = []
  current = []
  current_dur = 0.0
  for word in words:
    word_dur = word['duration_sec']
    if current and current_dur + word_dur > target_segment_sec:
      groups.append(current)
      current = [word]
      current_dur = word_dur
    else:
      current.append(word)
      current_dur += word_dur
  if current:
    groups.append(current)

  final_groups = []
  for group in groups:
    group_dur = sum(x['duration_sec'] for x in group)
    if group_dur <= max_segment_sec:
      final_groups.append(group)
      continue

    sub = []
    sub_dur = 0.0
    for word in group:
      if sub and sub_dur + word['duration_sec'] > max_segment_sec:
        final_groups.append(sub)
        sub = [word]
        sub_dur = word['duration_sec']
      else:
        sub.append(word)
        sub_dur += word['duration_sec']
    if sub:
      final_groups.append(sub)

  for seg_i,group in enumerate(final_groups):
    if total_segments >= max_segments:
      break

    text = ' '.join(x['word'] for x in group)
    start = group[0]['start_sec']
    end = group[-1]['end_sec']
    seg_dur = end - start
    conf = float(sum(x['score'] for x in group) / len(group))

    if conf < min_word_confidence:
      continue
    if seg_dur < min_segment_sec and len(final_groups) > 1:
      continue

    cut_start = max(0.0,start-padding_sec)
    cut_end = min(utt_dur,end+padding_sec)
    sample_start = int(cut_start * asr_sample_rate)
    sample_end = min(int(cut_end * asr_sample_rate),waveform.shape[-1])
    if sample_end <= sample_start:
      continue

    clip = waveform[sample_start:sample_end]
    segment_id = f'{uid}_seg_{seg_i:04d}'
    wav_path = seg_wav_dir / f'{segment_id}.wav'
    meta_path = seg_meta_dir / f'{segment_id}.json'

    if not wav_path.exists() or not meta_path.exists():
      save_wav(wav_path,clip,asr_sample_rate)
      meta = {
        'segment_id':segment_id,
        'utterance_id':uid,
        'text':text,
        'words':group,
        'start_sec':start,
        'end_sec':end,
        'duration_sec':duration(clip,asr_sample_rate),
        'padding_sec':padding_sec,
        'avg_confidence':conf,
        'sample_rate':asr_sample_rate,
        'num_samples':int(clip.shape[-1]),
        'wav_path':str(wav_path.relative_to(run_root)),
        'source_dataset':dataset_name,
        'source_row_index':record['row_index'],
      }
      write_json(meta_path,meta)

    total_segments += 1

n_segments = rebuild_jsonl(seg_meta_dir,run_root / 'segments' / 'metadata.jsonl')
print(f'segments this pass = {total_segments}')
print(f'segment metadata records = {n_segments}')


segments this pass = 514
segment metadata records = 514


### EnCodec RVQ tokens

In [12]:
codec_model = EncodecModel.encodec_model_24khz()
codec_model.set_target_bandwidth(encodec_bandwidth)
codec_model.to(DEVICE)
codec_model.eval()

print('encodec model = encodec_24khz')
print('bandwidth =',encodec_bandwidth)
print('sample rate =',codec_model.sample_rate)
print('channels =',codec_model.channels)


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


encodec model = encodec_24khz
bandwidth = 6.0
sample rate = 24000
channels = 1


In [13]:
tokenized = 0
reconstructed = 0
segment_files = sorted(seg_meta_dir.glob('*.json'))

for i,meta_path in enumerate(segment_files):
  if should_stop():
    print('time or disk limit hit during tokenization')
    break

  meta = read_json(meta_path)
  segment_id = meta['segment_id']
  token_path = codec_token_dir / f'{segment_id}.pt'
  recon_path = codec_recon_dir / f'{segment_id}.wav'
  codec_meta_path = codec_meta_dir / f'{segment_id}.json'

  if token_path.exists() and recon_path.exists() and codec_meta_path.exists():
    tokenized += 1
    reconstructed += 1
    continue

  wav_path = seg_wav_dir / f'{segment_id}.wav'
  if not wav_path.exists():
    continue

  wav,sr = torchaudio.load(str(wav_path))
  wav = convert_audio(wav,sr,codec_model.sample_rate,codec_model.channels)
  wav = wav.unsqueeze(0).to(DEVICE)

  with torch.no_grad():
    encoded_frames = codec_model.encode(wav)
    recon = codec_model.decode(encoded_frames)

  codes = torch.cat([frame[0] for frame in encoded_frames],dim=-1).detach().cpu()
  recon_audio = recon[0].detach().cpu()

  scales = []
  for frame in encoded_frames:
    if len(frame) > 1 and frame[1] is not None:
      scales.append(frame[1].detach().cpu())

  token_obj = {
    'codes':codes,
    'bandwidth':encodec_bandwidth,
    'sample_rate':codec_model.sample_rate,
    'channels':codec_model.channels,
    'segment_id':segment_id,
    'segment_text':meta.get('text',''),
    'token_shape':list(codes.shape),
    'batch_size':int(codes.shape[0]),
    'num_codebooks':int(codes.shape[1]),
    'token_frames':int(codes.shape[-1]),
    'model_name':'encodec_24khz',
  }
  if scales:
    token_obj['scales'] = torch.cat(scales,dim=-1)

  torch.save(token_obj,str(token_path))
  save_wav(recon_path,recon_audio,codec_model.sample_rate)

  old_stats = audio_stats(wav[0].detach().cpu())
  new_stats = audio_stats(recon_audio)
  codec_meta = {
    'segment_id':segment_id,
    'token_path':str(token_path.relative_to(run_root)),
    'reconstruction_path':str(recon_path.relative_to(run_root)),
    'token_shape':list(codes.shape),
    'batch_size':int(codes.shape[0]),
    'num_codebooks':int(codes.shape[1]),
    'token_frames':int(codes.shape[-1]),
    'bandwidth':encodec_bandwidth,
    'codec_sample_rate':codec_model.sample_rate,
    'codec_channels':codec_model.channels,
    'original_duration_sec':duration(wav[0].detach().cpu(),codec_model.sample_rate),
    'reconstruction_duration_sec':duration(recon_audio,codec_model.sample_rate),
    'original_rms':old_stats['rms'],
    'reconstruction_rms':new_stats['rms'],
    'original_peak':old_stats['peak'],
    'reconstruction_peak':new_stats['peak'],
  }
  write_json(codec_meta_path,codec_meta)

  tokenized += 1
  reconstructed += 1
  if (i+1)%50 == 0:
    print(f'{i+1}/{len(segment_files)} tokenized {segment_id}')

n_codec = rebuild_jsonl(codec_meta_dir,run_root / 'codec' / 'metadata.jsonl')
print(f'tokenized this pass = {tokenized}')
print(f'reconstructions this pass = {reconstructed}')
print(f'codec metadata records = {n_codec}')


50/514 tokenized 2035-147961-0018_seg_0000
100/514 tokenized 2035-152373-0006_seg_0001
150/514 tokenized 2086-149220-0001_seg_0000
200/514 tokenized 2086-149220-0017_seg_0001
250/514 tokenized 2086-149220-0048_seg_0001
300/514 tokenized 2277-149896-0005_seg_0001
350/514 tokenized 2277-149897-0005_seg_0002
400/514 tokenized 7976-105575-0004_seg_0000
450/514 tokenized 7976-110124-0004_seg_0000
500/514 tokenized 7976-110523-0010_seg_0000
tokenized this pass = 514
reconstructions this pass = 514
codec metadata records = 514


### Report

In [14]:
n_segments = rebuild_jsonl(seg_meta_dir,run_root / 'segments' / 'metadata.jsonl')
n_codec = rebuild_jsonl(codec_meta_dir,run_root / 'codec' / 'metadata.jsonl')

failed_records = []
for file in sorted(align_dir.glob('*.json')):
  item = read_json(file)
  if item.get('status') == 'failed':
    failed_records.append({
      'utterance_id':item['utterance_id'],
      'reason':item.get('error_reason','unknown'),
    })
(report_dir / 'failed_utterances.jsonl').write_text('\n'.join(json.dumps(x,ensure_ascii=False) for x in failed_records) + ('\n' if failed_records else ''))

print('segment metadata =',n_segments)
print('codec metadata =',n_codec)
print('failed utterances =',len(failed_records))


segment metadata = 514
codec metadata = 514
failed utterances = 0


In [15]:
alignment_files = list(align_dir.glob('*.json'))
num_aligned = sum(1 for p in alignment_files if read_json(p).get('status') == 'aligned')
num_failed = sum(1 for p in alignment_files if read_json(p).get('status') == 'failed')
num_segments = len(list(seg_meta_dir.glob('*.json')))
num_tokenized = len(list(codec_token_dir.glob('*.pt')))
num_recon = len(list(codec_recon_dir.glob('*.wav')))

segment_meta = [read_json(p) for p in sorted(seg_meta_dir.glob('*.json'))]
segment_durations = [x.get('duration_sec',0.0) for x in segment_meta]
segment_conf = [x.get('avg_confidence',0.0) for x in segment_meta]

total_seg_sec = sum(segment_durations)
avg_seg_sec = total_seg_sec / len(segment_durations) if segment_durations else 0.0
avg_conf = sum(segment_conf) / len(segment_conf) if segment_conf else 0.0
stopped_due_to_time = time_over()
stopped_due_to_disk = disk_low()

summary = {
  'run_name':run_name,
  'run_profile':run_profile,
  'completed':not stopped_due_to_time and not stopped_due_to_disk,
  'elapsed_hours':round(elapsed_hours(),4),
  'max_runtime_hours':max_runtime_hours,
  'dataset_name':dataset_name,
  'dataset_config':dataset_config,
  'dataset_split':dataset_split,
  'dataset_load_split':dataset_load_split,
  'dataset_streaming':dataset_streaming,
  'wav2vec2_model_name':wav2vec2_model_name,
  'encodec_model_name':'encodec_24khz',
  'encodec_bandwidth':encodec_bandwidth,
  'utterances_requested':max_utterances,
  'utterances_seen':len(manifest),
  'utterances_aligned':num_aligned,
  'utterances_failed':num_failed,
  'segments_extracted':num_segments,
  'segments_tokenized':num_tokenized,
  'reconstructions_saved':num_recon,
  'total_segment_duration_sec':round(total_seg_sec,2),
  'total_segment_duration_min':round(total_seg_sec/60,2),
  'average_segment_duration_sec':round(avg_seg_sec,4),
  'average_alignment_confidence':round(avg_conf,6),
  'min_segment_sec':round(min(segment_durations),4) if segment_durations else 0.0,
  'max_segment_sec':round(max(segment_durations),4) if segment_durations else 0.0,
  'target_segment_sec':target_segment_sec,
  'padding_sec':padding_sec,
  'min_word_confidence':min_word_confidence,
  'stopped_due_to_time_budget':stopped_due_to_time,
  'stopped_due_to_disk_budget':stopped_due_to_disk,
  'local_disk_free_gb':round(disk_free_gb('/content'),3),
  'min_free_disk_gb':min_free_disk_gb,
  'hf_cache':os.environ.get('HF_HOME'),
  'torch_hub_cache':torch.hub.get_dir(),
}
write_json(report_dir / 'summary.json',summary)
print(json.dumps(summary,indent=2))


{
  "run_name": "portfolio_t4_20260511_185544",
  "run_profile": "portfolio_t4",
  "completed": true,
  "elapsed_hours": 0.0467,
  "max_runtime_hours": 5.0,
  "dataset_name": "openslr/librispeech_asr",
  "dataset_config": "clean",
  "dataset_split": "validation",
  "dataset_load_split": "validation",
  "dataset_streaming": true,
  "wav2vec2_model_name": "facebook/wav2vec2-base-960h",
  "encodec_model_name": "encodec_24khz",
  "encodec_bandwidth": 6.0,
  "utterances_requested": 300,
  "utterances_seen": 300,
  "utterances_aligned": 300,
  "utterances_failed": 0,
  "segments_extracted": 514,
  "segments_tokenized": 514,
  "reconstructions_saved": 514,
  "total_segment_duration_sec": 1643.62,
  "total_segment_duration_min": 27.39,
  "average_segment_duration_sec": 3.1977,
  "average_alignment_confidence": 0.831545,
  "min_segment_sec": 0.5006,
  "max_segment_sec": 5.6526,
  "target_segment_sec": 3.0,
  "padding_sec": 0.05,
  "min_word_confidence": 0.01,
  "stopped_due_to_time_budget": fal

In [16]:
plots_saved = 0
for file in sorted(align_dir.glob('*.json')):
  if plots_saved >= 5:
    break
  item = read_json(file)
  if item.get('status') != 'aligned':
    continue
  words = item.get('word_spans',[])
  if not words:
    continue

  fig,ax = plt.subplots(figsize=(12,3))
  for word in words:
    start = word['start_sec']
    end = word['end_sec']
    ax.barh(0,end-start,left=start,height=0.6,color='steelblue',edgecolor='white')
    ax.text((start+end)/2,0,word['word'],ha='center',va='center',fontsize=9,color='white',fontweight='bold')
  ax.set_xlabel('Time (seconds)')
  ax.set_yticks([])
  ax.set_title(f"Word Alignment: {item['utterance_id']}")
  ax.set_xlim(left=0)
  plt.tight_layout()
  fig.savefig(str(sample_dir / f"alignment_{item['utterance_id']}.png"),dpi=100)
  plt.close(fig)
  plots_saved += 1

orig_saved = 0
recon_saved = 0
for file in sorted(seg_meta_dir.glob('*.json')):
  if orig_saved >= 5 and recon_saved >= 5:
    break
  item = read_json(file)
  sid = item['segment_id']
  wav_path = seg_wav_dir / f'{sid}.wav'
  recon_path = codec_recon_dir / f'{sid}.wav'
  if orig_saved < 5 and wav_path.exists():
    shutil.copy2(str(wav_path),str(sample_dir / f'original_{sid}.wav'))
    orig_saved += 1
  if recon_saved < 5 and recon_path.exists():
    shutil.copy2(str(recon_path),str(sample_dir / f'reconstruction_{sid}.wav'))
    recon_saved += 1

print('alignment plots saved =',plots_saved)
print('original audio samples saved =',orig_saved)
print('reconstruction samples saved =',recon_saved)


alignment plots saved = 5
original audio samples saved = 5
reconstruction samples saved = 5


In [17]:
complete_text = (
  'Pipeline complete.\n'
  f'completed at = {datetime.now().isoformat()}\n'
  f'elapsed hours = {elapsed_hours():.4f}\n'
)
if stopped_due_to_time:
  complete_text += 'Stopped due to time budget.\n'
if stopped_due_to_disk:
  complete_text += 'Stopped because local disk free space went below the configured limit.\n'
(run_root / 'RUN_COMPLETE.txt').write_text(complete_text)

cleanup_cache()
show_disk('end')

print('Pipeline complete.')
print('Raw audio-transcript pairs converted to word-aligned speech segments.')
print('Segments tokenized with EnCodec RVQ.')
print('Reconstructions and metadata saved to:')
print(run_root)


end: local disk used 43.52 GB / 112.64 GB, free 69.10 GB
Pipeline complete.
Raw audio-transcript pairs converted to word-aligned speech segments.
Segments tokenized with EnCodec RVQ.
Reconstructions and metadata saved to:
/content/drive/MyDrive/audioai/alignment_codec_runs/portfolio_t4_20260511_185544
